# 07 — Task 2C: Threat Model Extraction

Runs Prompt 2c (`prompts/prompt_2c_v2.txt`, unchanged from phase 1) on each of the 193 final scenarios. One LLM call per scenario; **the model sees only the scenario text**.

For each scenario it returns the 7 threat-model components (threat source, objective or harmful outcome, capability, knowledge, access, constraints or enabling conditions, target or asset at risk), each with **value**, **evidence** and an **uncertainty flag** (Clear / Ambiguous / Not specified).

**Input:** `output/scenarios_final.json` (the 193 scenarios of the phase-1 library: paper_id, scenario_id, scenario_text; built by `uv run python -m src.scripts.collect_scenarios`)
**Output:** `output/<model_name>/07_task_2c/`, e.g. `output/kimi-k3/07_task_2c/`
- `outputs/<paper_id>_<scenario_id>.json`: the model's answer per scenario
- `review/`, `summary.csv`, `consolidated.json`: built by the review cell
- `config.json`, `run_metadata.json`, `run_history/`, `logs/`: settings, run records, raw replies

**Models** (presets in `src/config.py`; pick one with `MODEL` below). The task settings are the same for every model, so only the model changes.

**Keys are never typed into the notebook:** locally in `.env` (git-ignored), in Colab as secrets (key icon), plus `GITHUB_TOKEN` in Colab.

**Start with `DRY_RUN = True`** (fake replies, no key, no cost; results go to `output/<model_name>_dryrun/`). Then `DRY_RUN = False` with `SCENARIO_KEYS = ["amodei_2026_S1"]` to test one real scenario. Finally `SCENARIO_KEYS = None` for all 193; finished scenarios are skipped, so re-running retries only failed ones.

In [ ]:
# Setup: works locally and in Google Colab
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_SLUG = "AI-Loc-Research/loss-of-control-definition-v2"

if IN_COLAB:
    from google.colab import userdata

    REPO_DIR = "/content/loss-of-control-definition-v2"
    github_token = userdata.get("GITHUB_TOKEN")  # private repo: a token with read (and write, to push) access

    def git(*args, cwd=None):
        result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
        if result.returncode != 0:  # never print the token
            raise RuntimeError("git failed: " + result.stderr.replace(github_token, "***"))
        return result.stdout

    if os.path.exists(REPO_DIR):
        git("pull", cwd=REPO_DIR)
    else:
        git("clone", f"https://{github_token}@github.com/{REPO_SLUG}.git", REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
                   check=True)
else:
    # Locally this notebook lives in <repo>/notebooks
    REPO_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

    # Load keys from <repo>/.env into environment variables (already-set variables win)
    from dotenv import load_dotenv
    load_dotenv(os.path.join(REPO_DIR, ".env"))

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo: {REPO_DIR}   (Colab: {IN_COLAB})")


In [ ]:
# Run settings: pick a model preset (src/config.py) and dry run or real run
from src.config import MODELS, get_config

MODEL = "kimi-k3-nvidia"   # any key of MODELS, e.g. "kimi-k3-nvidia", "kimi-k2.6-nvidia"
DRY_RUN = True             # True: fake model replies, no key, no cost. Set to False for the real run.
SCENARIO_KEYS = None       # None = all 193 scenarios; e.g. ["amodei_2026_S1"] to test one first

config = get_config("2c", MODEL)   # task 2C settings (prompt_2c_v2) + the model preset
config


In [ ]:
# API key for the chosen model's backend: read from Colab secrets or .env, never typed here
from src.config import api_key_name

KEY_NAME = api_key_name(config)    # e.g. "NVIDIA_API_KEY"
if IN_COLAB:
    try:
        API_KEY = userdata.get(KEY_NAME)          # Colab: key icon -> secret with this name
    except Exception:                             # secret missing, or notebook access not granted
        API_KEY = ""
else:
    API_KEY = os.environ.get(KEY_NAME, "")        # Local: from .env (or a set environment variable)

print(f"{KEY_NAME} found" if API_KEY else f"No {KEY_NAME}: only DRY_RUN = True will work")


In [ ]:
# Run Task 2C. Scenarios that already have output are skipped, so re-running resumes.
# Failed, refused or cut-off answers are listed as errors in run_metadata.json and retried on the next run.
from src.run_stage import run_2c

run_2c(config, API_KEY, repo_root=REPO_DIR, dry_run=DRY_RUN, scenario_keys=SCENARIO_KEYS)


In [ ]:
# Review docs (review/*.md), summary.csv and consolidated.json for this model
from src.generate_review import generate_review_docs_2c

generate_review_docs_2c(config, repo_root=REPO_DIR, dry_run=DRY_RUN)


In [ ]:
# Summary table: components specified / Clear / Ambiguous per scenario
import pandas as pd
from src.paths import task_output_dir

OUT_DIR = task_output_dir(REPO_DIR, config["model"], "07_task_2c", dry_run=DRY_RUN)
print("Results in:", OUT_DIR)
pd.read_csv(os.path.join(OUT_DIR, "summary.csv"))


In [ ]:
# Colab only: push the results to GitHub (Colab deletes its files when the session ends).
# Locally, commit and push with git as usual instead.
GIT_NAME = "Your Name"            # who the commit is from
GIT_EMAIL = "you@example.com"

if IN_COLAB and not DRY_RUN:
    results = os.path.relpath(OUT_DIR, REPO_DIR)
    git("config", "user.name", GIT_NAME, cwd=REPO_DIR)
    git("config", "user.email", GIT_EMAIL, cwd=REPO_DIR)
    git("add", results, cwd=REPO_DIR)
    git("commit", "-m", f"results: task 2C, {config['model']}", cwd=REPO_DIR)
    git("pull", "--rebase", cwd=REPO_DIR)
    git("push", cwd=REPO_DIR)
    print("Pushed", results)
else:
    print("Nothing pushed (dry run, or running locally).")
